# Homework 2 | Exploring Options | Tasks 1-4 due 09/22/25 | Task 5 due 09/24/25

Now that you have all of the options data stored locally on your computer and in pkl files, we can begin working with it. Your task will be to edit the monthly data and make some observations about its structure. See below.

## Tasks

### 1. For each file, remove the following columns: ['ImpliedVolatility', 'Delta','Gamma', 'Vega', 'Theta']

### 2. Add a column for the SPX index. MAKE SURE THE DATES MATCH! Hint: You might want to set the index of the options data to the date column... If yFinance doesn't let you download the SPX data skip this for now.

### 2. On paper or using markdown, use your knowledge of calculus to compute explicit formulas for the Greeks we discussed

### 3. Add columns of the Greeks using your formulae

### 4. Using the Newton Raphson method discussed on 9/17/25, calculate implied volatilies for each option and add this as a column to the data

### 5. Now plot the following and keep an eye out for specific relationships. We will talk about what we notice.

### 5.1 Using the ATM Strike and fixed date, plot the following:
* Call delta vs. time to maturity
* Call gamma vs. time to maturity
* Call theta vs. time to maturity
* Call vega vs. time to maturity
* Call implied volatility vs. time to maturity

Do the same for puts. What do you notice about the greeks for calls and puts?

### 5.2 From the span of 2015-2020:
* Calculate a 5,21,63,129 rolling realiezd volatilities for SPX
* Plot the implied volatilties for ATM options expiring at roughly the same 5,21,63,129 date marks. What I mean by this is, iterate through the data and calculate the implied volatility for ATM calls expriing in those times for every single day. You won't be using the same option for the 5 years if you get what I mean... Maybe it's clear already, and I'm being dramatic.
* Make some plots of IV - RV. What do you notice? When does the graph become positive?

### 5.3 Pick a date and maturity of your own choice and plot the following:
* Strike vs. delta
* Strike vs. theta
* Strike vs. IV
* Strike vs. option price

Feel free to do any additional analysis with the data at any time by the way. We are getting familiar with how options work here!

## Your work starts here

In [14]:
import yfinance as yf

In [36]:
import os
import re
import zipfile
import pickle
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import timedelta
import math
from scipy.special import erf

In [2]:
#PATHS
zip_folder = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX"
pickle_folder = os.path.join(zip_folder, "pickles")
os.makedirs(pickle_folder, exist_ok=True)

In [3]:
#Helper functions
def parse_date_from_name(name: str) -> str:
    """
    Return 'MMDDYYYY' from the filename.
    - Prefer 8-digit YYYYMMDD if present.
    - Else accept 6-digit YYYYMM and assume day=01.
    """
    m8 = re.search(r"(?<!\d)(20|19)\d{6}(?!\d)", name)
    if m8:
        dt = datetime.strptime(m8.group(), "%Y%m%d")
        return dt.strftime("%m%d%Y")
    m6 = re.search(r"(?<!\d)(20|19)\d{4}(?!\d)", name)
    if m6:
        dt = datetime.strptime(m6.group() + "01", "%Y%m%d")
        return dt.strftime("%m%d%Y")
    # fallback: return raw name stem
    return os.path.splitext(os.path.basename(name))[0]


In [4]:
def read_table_safely(path_or_buf) -> pd.DataFrame:
    """
    Try to read unknown .txt/.csv:
      1) Automatic delimiter inference (sep=None, engine='python')
      2) Try common delimiters
      3) Try whitespace
      4) Fallback to fixed-width
    """
    try:
        df = pd.read_csv(path_or_buf, sep=None, engine="python")
        if df.shape[1] > 1:
            return df
    except Exception:
        pass
    for sep in [",", "\t", "|", ";"]:
        try:
            df = pd.read_csv(path_or_buf, sep=sep)
            if df.shape[1] > 1:
                return df
        except Exception:
            continue
    try:
        df = pd.read_csv(path_or_buf, delim_whitespace=True)
        if df.shape[1] > 1:
            return df
    except Exception:
        pass
    return pd.read_fwf(path_or_buf)

In [5]:
def convert_zip_to_pickle():
    """
    For each zip in zip_folder:
      - open first .txt/.csv inside
      - parse into DataFrame
      - save as SPX_OPTIONS_MMDDYYYY.pkl in pickle_folder
    """
    converted = 0
    for file in os.listdir(zip_folder):
        if not file.lower().endswith(".zip"):
            continue
        zip_path = os.path.join(zip_folder, file)

        # derive target pickle name
        date_token = parse_date_from_name(file)
        target_name = f"SPX_OPTIONS_{date_token}.pkl"
        dst = os.path.join(pickle_folder, target_name)

        if os.path.exists(dst):
            print(f"[skip] {target_name} already exists")
            continue

        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                members = [m for m in zf.namelist() if m.lower().endswith((".txt", ".csv"))]
                if not members:
                    print(f"[warn] {file} contains no .txt/.csv")
                    continue
                # use the first file found
                with zf.open(members[0]) as f:
                    df = read_table_safely(f)

            # Save to pickle
            with open(dst, "wb") as out:
                pickle.dump(df, out, protocol=pickle.HIGHEST_PROTOCOL)

            converted += 1
            print(f"Converted {file} -> {target_name} (rows={len(df)}, cols={df.shape[1]})")
        except Exception as e:
            print(f"[error] Could not process {file}: {e}")

    if converted == 0:
        print("No zip files converted.")

In [6]:
def open_sample_pickle():
    """Peek at one pickle to confirm success."""
    pkl_files = sorted([f for f in os.listdir(pickle_folder) if f.lower().endswith(".pkl")])
    if not pkl_files:
        print("No pickle files found.")
        return
    sample = os.path.join(pickle_folder, pkl_files[0])
    with open(sample, "rb") as f:
        df = pickle.load(f)
    print(f"\nOpened sample: {os.path.basename(sample)}")
    print(f"Shape: {df.shape}")
    print("\nColumns:", df.columns.tolist()[:15], "..." if len(df.columns) > 15 else "")
    print(df.head())

In [7]:
if __name__ == "__main__":
    convert_zip_to_pickle()
    open_sample_pickle()

Converted IVYOPPRCD_200704.zip -> SPX_OPTIONS_04012007.pkl (rows=16684, cols=23)
Converted IVYOPPRCD_202107.zip -> SPX_OPTIONS_07012021.pkl (rows=421086, cols=23)
Converted IVYOPPRCD_200710.zip -> SPX_OPTIONS_10012007.pkl (rows=22466, cols=23)
Converted IVYOPPRCD_199610.zip -> SPX_OPTIONS_10011996.pkl (rows=7495, cols=23)
Converted IVYOPPRCD_199604.zip -> SPX_OPTIONS_04011996.pkl (rows=6016, cols=23)
Converted IVYOPPRCD_201803.zip -> SPX_OPTIONS_03012018.pkl (rows=266624, cols=23)
Converted IVYOPPRCD_200506.zip -> SPX_OPTIONS_06012005.pkl (rows=12880, cols=23)
Converted IVYOPPRCD_200512.zip -> SPX_OPTIONS_12012005.pkl (rows=13327, cols=23)
Converted IVYOPPRCD_202305.zip -> SPX_OPTIONS_05012023.pkl (rows=384004, cols=23)
Converted IVYOPPRCD_201208.zip -> SPX_OPTIONS_08012012.pkl (rows=64938, cols=23)
Converted IVYOPPRCD_200102.zip -> SPX_OPTIONS_02012001.pkl (rows=9508, cols=23)
Converted IVYOPPRCD_200103.zip -> SPX_OPTIONS_03012001.pkl (rows=10897, cols=23)
Converted IVYOPPRCD_201209.z

## We just converted the zip files to pickle

In [3]:
# For each file, remove the columns stated above
# Path to your pickle folder
pickle_folder = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles"

# Columns to remove
DROP_COLS = ['ImpliedVolatility', 'Delta', 'Gamma', 'Vega', 'Theta']

def clean_pickles():
    for file in os.listdir(pickle_folder):
        if not file.lower().endswith(".pkl"):
            continue
        path = os.path.join(pickle_folder, file)
        try:
            # Load DataFrame
            with open(path, "rb") as f:
                df = pickle.load(f)

            # Drop columns if they exist
            df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")

            # Save back to pickle
            with open(path, "wb") as f:
                pickle.dump(df, f, protocol=pickle.HIGHEST_PROTOCOL)

            print(f"Cleaned {file} (rows={len(df)}, cols={df.shape[1]})")

        except Exception as e:
            print(f"[error] Could not process {file}: {e}")

if __name__ == "__main__":
    clean_pickles()

Cleaned SPX_OPTIONS_11012005.pkl (rows=13385, cols=18)
Cleaned SPX_OPTIONS_11012011.pkl (rows=42962, cols=18)
Cleaned SPX_OPTIONS_03012016.pkl (rows=171916, cols=18)
Cleaned SPX_OPTIONS_03012002.pkl (rows=9650, cols=18)
Cleaned SPX_OPTIONS_12012019.pkl (rows=334195, cols=18)
Cleaned SPX_OPTIONS_06012019.pkl (rows=285786, cols=18)
Cleaned SPX_OPTIONS_05012011.pkl (rows=42310, cols=18)
Cleaned SPX_OPTIONS_05012005.pkl (rows=12780, cols=18)
Cleaned SPX_OPTIONS_08012012.pkl (rows=64938, cols=18)
Cleaned SPX_OPTIONS_08012006.pkl (rows=14982, cols=18)
Cleaned SPX_OPTIONS_08012007.pkl (rows=21028, cols=18)
Cleaned SPX_OPTIONS_08012013.pkl (rows=76296, cols=18)
Cleaned SPX_OPTIONS_05012004.pkl (rows=10922, cols=18)
Cleaned SPX_OPTIONS_05012010.pkl (rows=37835, cols=18)
Cleaned SPX_OPTIONS_06012018.pkl (rows=266888, cols=18)
Cleaned SPX_OPTIONS_12012018.pkl (rows=268732, cols=18)
Cleaned SPX_OPTIONS_03012003.pkl (rows=10816, cols=18)
Cleaned SPX_OPTIONS_03012017.pkl (rows=187554, cols=18)
Clean

In [17]:
pickle_folder = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles"
pickle_out_folder = os.path.join(pickle_folder, "with_spx")
os.makedirs(pickle_out_folder, exist_ok=True)

SPX_COL = "SPX_AdjClose"
spx_path = "/Users/zakdjahed/Desktop/TAMID/spx_1990_2024"  # <-- this is your CSV file

DATE_CANDIDATES = [
    "date", "trade_date", "tradedate", "quote_date", "quotedate",
    "DataDate", "DataAsOf", "asofdate"
]

# --- HELPERS ---

def find_date_col(df: pd.DataFrame) -> str:
    m = {c.lower(): c for c in df.columns}
    for cand in DATE_CANDIDATES:
        if cand in m: return m[cand]
    for c in df.columns:
        if "date" in c.lower(): return c
    raise ValueError("Could not find a date column. Update DATE_CANDIDATES.")

def normalize_to_date_index(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    dt = pd.to_datetime(df[date_col], errors="coerce")
    try:
        dt = dt.dt.tz_localize(None)
    except Exception:
        pass
    d = pd.to_datetime(dt.dt.date)  # normalize to day
    out = df.copy()
    out.index = d
    return out

def load_spx_csv(path: str) -> pd.DataFrame:
    """
    Load the odd Yahoo-like CSV you showed:
      Row1: Price,Close,High,Low,Open,Volume
      Row2: Ticker,^SPX,^SPX,^SPX,^SPX,^SPX
      Row3: Date,,,,,
      Rows: YYYY-MM-DD,<numbers...>

    Returns a DataFrame indexed by date with one column SPX_AdjClose.
    """
    if not os.path.isfile(path):
        raise FileNotFoundError(f"SPX CSV not found: {path}")

    # First pass: peek one data line to count columns
    with open(path, "r", encoding="utf-8") as f:
        # skip 3 header rows
        for _ in range(3):
            next(f, None)
        sample = next(f, "").rstrip("\n")
    if not sample:
        raise ValueError("SPX CSV has no data rows after headers.")
    fields = sample.split(",")
    n_cols = len(fields)

    # Expect at least Date + 2 columns (Price/Close) + Volume
    if n_cols < 3:
        raise ValueError(f"Unexpected number of columns in data row: {n_cols}")

    # Build column names dynamically: first is Date, then as many as present from canonical list
    canonical = ["Price", "Close", "High", "Low", "Open", "Volume"]
    names = ["Date"] + canonical[: n_cols - 1]

    # Now read the whole file skipping first 3 header rows
    spx = pd.read_csv(
        path,
        skiprows=3,
        header=None,
        names=names,
        dtype={0: str},  # Date as string first
    )

    # Parse Date -> daily DatetimeIndex
    spx["Date"] = pd.to_datetime(spx["Date"], errors="coerce")
    spx = spx.dropna(subset=["Date"]).copy()
    spx["Date"] = pd.to_datetime(spx["Date"].dt.date)  # normalize to day
    spx = spx.set_index("Date").sort_index()

    # Coerce numeric columns
    for c in names[1:]:
        if c in spx.columns:
            spx[c] = pd.to_numeric(spx[c], errors="coerce")

    # Choose adjusted close proxy: prefer 'Price', else 'Close'
    if "Price" in spx.columns and spx["Price"].notna().any():
        px = spx["Price"]
    elif "Close" in spx.columns and spx["Close"].notna().any():
        px = spx["Close"]
    else:
        raise ValueError("Could not find usable Price or Close column in SPX CSV.")

    # Build output: one column SPX_AdjClose, drop duplicate dates if any
    out = px.to_frame(name=SPX_COL)
    out = out[~out.index.duplicated(keep="last")]
    return out


# --- MAIN ---

def add_spx_to_pickles(strict=True):
    pkl_files = sorted([f for f in os.listdir(pickle_folder) if f.lower().endswith(".pkl")])
    if not pkl_files:
        raise RuntimeError("No .pkl files found in pickle_folder.")
    pkl_paths = [os.path.join(pickle_folder, f) for f in pkl_files]

    # Load SPX once from your CSV
    spx = load_spx_csv(spx_path)
    print(f"Loaded SPX rows: {len(spx)}; range {spx.index.min().date()} .. {spx.index.max().date()}")

    for path in pkl_paths:
        with open(path, "rb") as f:
            df = pickle.load(f)

        date_col = find_date_col(df)
        df_idx = normalize_to_date_index(df, date_col)

        merged = df_idx.join(spx, how="inner")  # exact date match only
        if strict and len(merged) != len(df_idx):
            print(f"[warn] {os.path.basename(path)}: {len(df_idx)-len(merged)} rows dropped (no exact SPX match).")

        out_path = os.path.join(pickle_out_folder, os.path.basename(path))
        with open(out_path, "wb") as f:
            pickle.dump(merged, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Saved: {os.path.basename(out_path)} (rows={len(merged)}, cols={merged.shape[1]})")

if __name__ == "__main__":
    add_spx_to_pickles(strict=True)

Loaded SPX rows: 8565; range 1990-01-02 .. 2023-12-29
[warn] SPX_OPTIONS_01011996.pkl: 5580 rows dropped (no exact SPX match).
Saved: SPX_OPTIONS_01011996.pkl (rows=0, cols=19)
[warn] SPX_OPTIONS_01011997.pkl: 8378 rows dropped (no exact SPX match).
Saved: SPX_OPTIONS_01011997.pkl (rows=0, cols=19)
[warn] SPX_OPTIONS_01011998.pkl: 8745 rows dropped (no exact SPX match).
Saved: SPX_OPTIONS_01011998.pkl (rows=0, cols=19)
[warn] SPX_OPTIONS_01011999.pkl: 8654 rows dropped (no exact SPX match).
Saved: SPX_OPTIONS_01011999.pkl (rows=0, cols=19)
[warn] SPX_OPTIONS_01012000.pkl: 10584 rows dropped (no exact SPX match).
Saved: SPX_OPTIONS_01012000.pkl (rows=0, cols=19)
[warn] SPX_OPTIONS_01012001.pkl: 10863 rows dropped (no exact SPX match).
Saved: SPX_OPTIONS_01012001.pkl (rows=0, cols=19)
[warn] SPX_OPTIONS_01012002.pkl: 9829 rows dropped (no exact SPX match).
Saved: SPX_OPTIONS_01012002.pkl (rows=0, cols=19)
[warn] SPX_OPTIONS_01012003.pkl: 10702 rows dropped (no exact SPX match).
Saved: SP

In [20]:
import pickle, os

test_pkl = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles/SPX_OPTIONS_01011996.pkl"

with open(test_pkl, "rb") as f:
    df0 = pickle.load(f)

# list all columns
print("Columns:", df0.columns.tolist())

# show first 20 values of anything with 'date' in its name
for c in df0.columns:
    if "date" in c.lower():
        print(f"\nColumn: {c}")
        print(df0[c].head(20).tolist())


Columns: ['SecurityID', 'Date', 'symbol', 'symbolflag', 'Strike', 'Expiration', 'CallPut', 'BestBid', 'BestOffer', 'LastTradeDate', 'Volume', 'OpenInterest', 'SpecialSettlement', 'OptionID2', 'AdjustmentFactor', 'AMSettlement', 'ContractSize', 'ExpiryIndicator']

Column: Date
[19960104, 19960105, 19960108, 19960109, 19960110, 19960111, 19960112, 19960115, 19960116, 19960117, 19960118, 19960119, 19960122, 19960123, 19960124, 19960125, 19960126, 19960129, 19960130, 19960131]

Column: LastTradeDate
[19960104.0, 19960105.0, 19960105.0, 19960109.0, 19960110.0, 19960111.0, 19960112.0, 19960115.0, 19960116.0, 19960117.0, 19960118.0, 19960119.0, 19960122.0, 19960123.0, 19960124.0, 19960125.0, 19960126.0, 19960129.0, 19960130.0, 19960131.0]


In [21]:
import os
import pickle
import numpy as np
import pandas as pd

# --- CONFIG ---
pickle_folder = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles"
pickle_out_folder = os.path.join(pickle_folder, "with_spx")
os.makedirs(pickle_out_folder, exist_ok=True)

SPX_COL = "SPX_AdjClose"
spx_path = "/Users/zakdjahed/Desktop/TAMID/spx_1990_2024"  # your CSV with 3-row header

DATE_CANDIDATES = [
    "Date", "date", "trade_date", "tradedate", "quote_date", "quotedate",
    "DataDate", "DataAsOf", "asofdate"
]

# ---------- robust parsers ----------

def parse_yyyymmdd(series: pd.Series) -> pd.Series:
    """Parse YYYYMMDD stored as int/float/str → datetime64[ns] normalized to date."""
    s = pd.to_numeric(series, errors="coerce")
    # Drop decimals like 19960104.0
    s = s.round(0).astype("Int64")
    dt = pd.to_datetime(s.astype(str), format="%Y%m%d", errors="coerce")
    return pd.to_datetime(dt.dt.date)

def parse_dates_generic(series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
    return pd.to_datetime(dt.dt.date)

def parse_option_dates(series: pd.Series) -> pd.Series:
    """Try YYYYMMDD first (your case), then generic fallback."""
    dt = parse_yyyymmdd(series)
    if dt.isna().mean() > 0.5:  # if over half failed, try generic
        dt2 = parse_dates_generic(series)
        dt = dt.fillna(dt2)
    return dt

def find_date_col(df: pd.DataFrame) -> str:
    for cand in DATE_CANDIDATES:
        if cand in df.columns:
            return cand
    # last resort: any column containing 'date'
    for c in df.columns:
        if "date" in c.lower():
            return c
    raise ValueError("No date-like column found.")

def normalize_to_date_index(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    dt = parse_option_dates(df[date_col])
    out = df.copy()
    out.index = dt
    return out

# ---------- SPX loader for your 3-row header CSV ----------

def load_spx_csv(path: str) -> pd.DataFrame:
    # Peek to count columns after 3 header rows
    with open(path, "r", encoding="utf-8") as f:
        for _ in range(3): next(f, None)
        sample = next(f, "").rstrip("\n")
    if not sample:
        raise ValueError("SPX CSV: no data rows after header.")
    n_cols = len(sample.split(","))
    canonical = ["Price","Close","High","Low","Open","Volume"]
    names = ["Date"] + canonical[: n_cols-1]
    spx = pd.read_csv(path, skiprows=3, header=None, names=names)
    spx["Date"] = pd.to_datetime(spx["Date"], errors="coerce").dt.date
    spx = spx.dropna(subset=["Date"]).copy()
    spx = spx.set_index(pd.to_datetime(spx["Date"])).drop(columns=["Date"])
    spx.index = pd.to_datetime(spx.index.date)  # normalize
    for c in spx.columns: spx[c] = pd.to_numeric(spx[c], errors="coerce")
    px_col = "Price" if "Price" in spx.columns and spx["Price"].notna().any() else "Close"
    out = spx[[px_col]].rename(columns={px_col: SPX_COL})
    return out[~out.index.duplicated(keep="last")]

# ---------- MAIN ----------

def add_spx_to_pickles(strict=False):  # left-join while validating
    pkl_files = sorted([f for f in os.listdir(pickle_folder) if f.lower().endswith(".pkl")])
    if not pkl_files:
        raise RuntimeError("No .pkl files found.")

    spx = load_spx_csv(spx_path)  # date-indexed, col = SPX_AdjClose
    print(f"Loaded SPX: {spx.index.min().date()} .. {spx.index.max().date()}  (rows={len(spx)})")

    for fname in pkl_files:
        path = os.path.join(pickle_folder, fname)
        with open(path, "rb") as f:
            df = pickle.load(f)

        try:
            date_col = find_date_col(df)
        except Exception as e:
            print(f"[skip] {fname}: {e}")
            continue

        df_idx = normalize_to_date_index(df, date_col)

        # Quick sanity logs
        if len(df_idx):
            print(f"{fname}: options {df_idx.index.min().date()} .. {df_idx.index.max().date()}  n={len(df_idx)}")

        merged = df_idx.join(spx, how="left")   # keep option rows; fill SPX when available
        matched = merged[SPX_COL].notna().sum()
        print(f"  matched {matched}/{len(merged)} rows to SPX")

        out_path = os.path.join(pickle_out_folder, fname)
        with open(out_path, "wb") as f:
            pickle.dump(merged, f, protocol=pickle.HIGHEST_PROTOCOL)

if __name__ == "__main__":
    add_spx_to_pickles(strict=False)


Loaded SPX: 1990-01-02 .. 2023-12-29  (rows=8565)
SPX_OPTIONS_01011996.pkl: options 1996-01-04 .. 1996-01-31  n=5580
  matched 5580/5580 rows to SPX
SPX_OPTIONS_01011997.pkl: options 1997-01-02 .. 1997-01-31  n=8378
  matched 8378/8378 rows to SPX
SPX_OPTIONS_01011998.pkl: options 1998-01-02 .. 1998-01-30  n=8745
  matched 8745/8745 rows to SPX
SPX_OPTIONS_01011999.pkl: options 1999-01-04 .. 1999-01-29  n=8654
  matched 8654/8654 rows to SPX
SPX_OPTIONS_01012000.pkl: options 2000-01-03 .. 2000-01-31  n=10584
  matched 10584/10584 rows to SPX
SPX_OPTIONS_01012001.pkl: options 2001-01-02 .. 2001-01-31  n=10863
  matched 10863/10863 rows to SPX
SPX_OPTIONS_01012002.pkl: options 2002-01-02 .. 2002-01-31  n=9829
  matched 9829/9829 rows to SPX
SPX_OPTIONS_01012003.pkl: options 2003-01-02 .. 2003-01-31  n=10702
  matched 10702/10702 rows to SPX
SPX_OPTIONS_01012004.pkl: options 2004-01-02 .. 2004-01-30  n=10358
  matched 10358/10358 rows to SPX
SPX_OPTIONS_01012005.pkl: options 2005-01-03 ..

In [8]:
# FILE WITH ALL RELEVANT SPX DATA - SHARE

import yfinance as yf 

spx_1990_2024 = yf.download("^SPX", start="1990-01-01", end ="2024-01-01")
spx_1990_2024.to_csv("spx_1990_2024")


/var/folders/8m/rcncq3bj137bqg7bk01pr0xw0000gn/T/ipykernel_8238/279610779.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spx_1990_2024 = yf.download("^SPX", start="1990-01-01", end ="2024-01-01")
[*********************100%***********************]  1 of 1 completed


In [10]:
pickle_folder = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles"

def peek_pickles(n=5):
    files = sorted([f for f in os.listdir(pickle_folder) if f.lower().endswith(".pkl")])
    if not files:
        print("No pickle files found.")
        return
    for fname in files[:3]:   # show just the first 3 files to avoid overload
        path = os.path.join(pickle_folder, fname)
        with open(path, "rb") as f:
            df = pickle.load(f)
        print(f"\n {fname} ")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(n))

if __name__ == "__main__":
    peek_pickles(n=5)


=== SPX_OPTIONS_01011996.pkl ===
Shape: (5580, 18)
Columns: ['SecurityID', 'Date', 'symbol', 'symbolflag', 'Strike', 'Expiration', 'CallPut', 'BestBid', 'BestOffer', 'LastTradeDate', 'Volume', 'OpenInterest', 'SpecialSettlement', 'OptionID2', 'AdjustmentFactor', 'AMSettlement', 'ContractSize', 'ExpiryIndicator']
   SecurityID      Date    symbol  symbolflag  Strike  Expiration CallPut  \
0      108105  19960104  098A3.1A           0  600000    19960316       C   
1      108105  19960105  098A3.1A           0  600000    19960316       C   
2      108105  19960108  098A3.1A           0  600000    19960316       C   
3      108105  19960109  098A3.1A           0  600000    19960316       C   
4      108105  19960110  098A3.1A           0  600000    19960316       C   

   BestBid  BestOffer  LastTradeDate  Volume  OpenInterest  SpecialSettlement  \
0   24.750     25.750     19960104.0     150          5633                  0   
1   24.625     25.375     19960105.0      50          5648  

In [27]:
import os
import pickle
import numpy as np
import pandas as pd

# --- CONFIG ---
ORIG_PICKLES = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles"  # point to NON-empty pickles
OUT_PICKLES  = os.path.join(ORIG_PICKLES, "with_spx")
os.makedirs(OUT_PICKLES, exist_ok=True)

SPX_CSV = "/Users/zakdjahed/Desktop/TAMID/spx_1990_2024"   # your CSV (no extension in name is fine)
SPX_COL = "SPX_AdjClose"

DATE_COL_CANDIDATES = [
    "Date","date","trade_date","tradedate","quote_date","quotedate","DataDate","DataAsOf","asofdate"
]

# --- Parsers ---

def parse_yyyymmdd(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").round(0).astype("Int64")
    dt = pd.to_datetime(s.astype(str), format="%Y%m%d", errors="coerce")
    return pd.to_datetime(dt.dt.date)

def parse_option_dates(series: pd.Series) -> pd.Series:
    dt = parse_yyyymmdd(series)
    if dt.isna().mean() > 0.5:  # fallback if needed
        dt2 = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
        dt = dt.fillna(pd.to_datetime(dt2.dt.date))
    return dt

def find_date_col(df: pd.DataFrame) -> str:
    for c in DATE_COL_CANDIDATES:
        if c in df.columns: return c
    for c in df.columns:
        if "date" in c.lower(): return c
    raise ValueError("No date-like column found.")

def normalize_to_date_index(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    dt = parse_option_dates(df[date_col])
    out = df.copy()
    out.index = dt
    return out

# --- SPX CSV loader (handles the 3-row header you showed) ---

def load_spx_csv(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        for _ in range(3): next(f, None)
        sample = next(f, "").rstrip("\n")
    if not sample:
        raise ValueError("SPX CSV has no data rows.")
    n_cols = len(sample.split(","))
    canonical = ["Price","Close","High","Low","Open","Volume"]
    names = ["Date"] + canonical[: n_cols-1]

    spx = pd.read_csv(path, skiprows=3, header=None, names=names)
    spx["Date"] = pd.to_datetime(spx["Date"], errors="coerce").dt.date
    spx = spx.dropna(subset=["Date"]).copy()
    spx = spx.set_index(pd.to_datetime(spx["Date"])).drop(columns=["Date"])
    spx.index = pd.to_datetime(spx.index.date)
    for c in spx.columns:
        spx[c] = pd.to_numeric(spx[c], errors="coerce")

    px_col = "Price" if "Price" in spx.columns and spx["Price"].notna().any() else "Close"
    out = spx[[px_col]].rename(columns={px_col: SPX_COL})
    out = out[~out.index.duplicated(keep="last")]
    return out

# --- Main merge ---

def add_spx_left_join():
    spx = load_spx_csv(SPX_CSV)
    print(f"SPX loaded: {spx.index.min().date()} .. {spx.index.max().date()}  (rows={len(spx)})")

    files = sorted([f for f in os.listdir(ORIG_PICKLES) if f.lower().endswith(".pkl")])
    if not files:
        print("No pickle files found.")
        return

    for f in files:
        src = os.path.join(ORIG_PICKLES, f)
        try:
            with open(src, "rb") as fh:
                df = pickle.load(fh)
        except Exception as e:
            print(f"[skip] {f}: could not read pickle ({e})")
            continue

        if len(df) == 0:
            print(f"[skip] {f}: source has 0 rows (likely from earlier inner-join overwrite).")
            continue

        try:
            date_col = find_date_col(df)
        except Exception as e:
            print(f"[skip] {f}: {e}")
            continue

        dfi = normalize_to_date_index(df, date_col)

        merged = dfi.join(spx, how="left")  # keep all option rows; fill SPX where available
        matched = merged[SPX_COL].notna().sum()
        print(f"{f}: matched {matched}/{len(merged)} rows")

        dst = os.path.join(OUT_PICKLES, f)
        with open(dst, "wb") as fh:
            pickle.dump(merged, fh, protocol=pickle.HIGHEST_PROTOCOL)

if __name__ == "__main__":
    add_spx_left_join()


SPX loaded: 1990-01-02 .. 2023-12-29  (rows=8565)
SPX_OPTIONS_01011996.pkl: matched 5580/5580 rows
SPX_OPTIONS_01011997.pkl: matched 8378/8378 rows
SPX_OPTIONS_01011998.pkl: matched 8745/8745 rows
SPX_OPTIONS_01011999.pkl: matched 8654/8654 rows
SPX_OPTIONS_01012000.pkl: matched 10584/10584 rows
SPX_OPTIONS_01012001.pkl: matched 10863/10863 rows
SPX_OPTIONS_01012002.pkl: matched 9829/9829 rows
SPX_OPTIONS_01012003.pkl: matched 10702/10702 rows
SPX_OPTIONS_01012004.pkl: matched 10358/10358 rows
SPX_OPTIONS_01012005.pkl: matched 11424/11424 rows
SPX_OPTIONS_01012006.pkl: matched 12733/12733 rows
SPX_OPTIONS_01012007.pkl: matched 14718/14718 rows
SPX_OPTIONS_01012008.pkl: matched 22576/22576 rows
SPX_OPTIONS_01012009.pkl: matched 30300/30300 rows
SPX_OPTIONS_01012010.pkl: matched 35271/35271 rows
SPX_OPTIONS_01012011.pkl: matched 39620/39620 rows
SPX_OPTIONS_01012012.pkl: matched 41966/41966 rows
SPX_OPTIONS_01012013.pkl: matched 59431/59431 rows
SPX_OPTIONS_01012014.pkl: matched 84604/84

In [30]:
# Sanity check to see if Spx_adjClose is there

import os, pickle

# Path to your output folder (where merged pickles live)
OUT_PICKLES = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles/with_spx"

# Grab one file to inspect
test_file = os.path.join(OUT_PICKLES, "SPX_OPTIONS_01011996.pkl")

with open(test_file, "rb") as f:
    df = pickle.load(f)

print("Columns:", df.columns.tolist()[:20])   # list first 20 columns
print("Has SPX_AdjClose?:", "SPX_AdjClose" in df.columns)

if "SPX_AdjClose" in df.columns:
    print("Non-null values:", df["SPX_AdjClose"].notna().sum())
    print(df[["Date", "Strike", "SPX_AdjClose"]].head(10))


Columns: ['SecurityID', 'Date', 'symbol', 'symbolflag', 'Strike', 'Expiration', 'CallPut', 'BestBid', 'BestOffer', 'LastTradeDate', 'Volume', 'OpenInterest', 'SpecialSettlement', 'OptionID2', 'AdjustmentFactor', 'AMSettlement', 'ContractSize', 'ExpiryIndicator', 'SPX_AdjClose']
Has SPX_AdjClose?: True
Non-null values: 5580
                Date  Strike  SPX_AdjClose
Date                                      
1996-01-04  19960104  600000    617.700012
1996-01-05  19960105  600000    616.710022
1996-01-08  19960108  600000    618.460022
1996-01-09  19960109  600000    609.450012
1996-01-10  19960110  600000    598.479980
1996-01-11  19960111  600000    602.690002
1996-01-12  19960112  600000    601.809998
1996-01-15  19960115  600000    599.820007
1996-01-16  19960116  600000    608.440002
1996-01-17  19960117  600000    606.369995


In [37]:
# Adding the Greeks (Delta, Gamma, Theta, and Vega) to the columns

# We assume that r = 0.045 and the dividend yield is 0

# We will have to compute the implied volatility based off of that

import os, pickle, numpy as np, pandas as pd

# -------- CONFIG --------
PICKLE_IN  = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles/with_spx"
PICKLE_OUT = os.path.join(PICKLE_IN, "with_iv_greeks")
os.makedirs(PICKLE_OUT, exist_ok=True)

R, Q = 0.045, 0.0
DAYS_PER_YEAR = 365.0
COL_S, COL_K, COL_D, COL_X, COL_CP = "SPX_AdjClose", "Strike", "Date", "Expiration", "CallPut"
COL_BID, COL_ASK = "BestBid", "BestOffer"

# ---- helpers ----
def to_dt_yyyymmdd(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").round(0).astype("Int64").astype(str)
    return pd.to_datetime(s, format="%Y%m%d", errors="coerce")

def phi(x): return np.exp(-0.5*x*x)/np.sqrt(2*np.pi)
def N(x):
    x = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))



def d1d2(S,K,T,r,q,sig):
    denom = sig*np.sqrt(T)
    with np.errstate(divide='ignore', invalid='ignore'):
        d1 = (np.log(S/K) + (r-q+0.5*sig*sig)*T)/denom
        d2 = d1 - denom
    bad = (S<=0)|(K<=0)|(T<=0)|(sig<=0)|~np.isfinite(S)|~np.isfinite(K)|~np.isfinite(T)|~np.isfinite(sig)
    d1 = np.where(bad, np.nan, d1); d2 = np.where(bad, np.nan, d2)
    return d1, d2

def bs_price(S,K,T,r,q,sig,is_call=True):
    d1,d2 = d1d2(S,K,T,r,q,sig); dq = np.exp(-q*T); dr = np.exp(-r*T)
    return np.where(is_call, S*dq*N(d1)-K*dr*N(d2), K*dr*N(-d2)-S*dq*N(-d1))

def vega(S,K,T,r,q,sig):
    d1,_ = d1d2(S,K,T,r,q,sig); return S*np.exp(-q*T)*phi(d1)*np.sqrt(T)

def iv_newton(price,S,K,T,r,q,is_call, it=100, tol_p=1e-6, smin=1e-6, smax=5.0):
    price = np.asarray(price,float); S=np.asarray(S,float); K=np.asarray(K,float)
    T=np.asarray(T,float); is_call=np.asarray(is_call,bool)
    dq=np.exp(-q*T); dr=np.exp(-r*T)
    intrinsic = np.where(is_call, np.maximum(S*dq-K*dr,0.0), np.maximum(K*dr-S*dq,0.0))
    extrinsic = np.maximum(price - intrinsic, 0.0)
    with np.errstate(divide='ignore', invalid='ignore'):
        sig = np.sqrt(2*np.pi/T) * (extrinsic/(S*dq))
    sig = np.clip(sig, smin, smax); sig[~np.isfinite(sig)] = np.nan

    for _ in range(it):
        mask = np.isfinite(sig)
        if not mask.any(): break
        model = bs_price(S[mask],K[mask],T[mask],r,q,sig[mask],is_call[mask])
        diff  = model - price[mask]
        if np.all(np.abs(diff) < tol_p): break
        v = vega(S[mask],K[mask],T[mask],r,q,sig[mask])
        upd = v > 1e-12
        if not np.any(upd): break
        sigm = sig[mask]
        sigm[upd] = np.clip(sigm[upd] - diff[upd]/v[upd], smin, smax)
        sig[mask] = sigm
    return sig

def greeks(S,K,T,r,q,sig,is_call):
    d1,d2 = d1d2(S,K,T,r,q,sig); dq=np.exp(-q*T); dr=np.exp(-r*T)
    Delta = np.where(is_call, dq*N(d1), -dq*N(-d1))
    Gamma = dq*phi(d1)/(S*sig*np.sqrt(T))
    Gamma = np.where((T<=0)|(sig<=0)|(S<=0), np.nan, Gamma)
    Theta_common = -(S*dq*phi(d1)*sig)/(2*np.sqrt(T))
    Theta_call = Theta_common - r*K*dr*N(d2) + q*S*dq*N(d1)
    Theta_put  = Theta_common + r*K*dr*N(-d2) - q*S*dq*N(-d1)
    Theta = np.where(is_call, Theta_call, Theta_put)
    Vega  = S*dq*phi(d1)*np.sqrt(T)
    return Delta, Gamma, Theta, Vega

def process_file(path):
    with open(path,"rb") as f: df = pickle.load(f)
    if COL_S not in df.columns:
        raise KeyError(f"Missing {COL_S} — are you reading from with_spx/?")
    # dates + T
    d  = to_dt_yyyymmdd(df[COL_D]); x = to_dt_yyyymmdd(df[COL_X])
    T  = (x - d).dt.days.astype(float)/DAYS_PER_YEAR
    # inputs
    S = pd.to_numeric(df[COL_S], errors="coerce")
    K = pd.to_numeric(df[COL_K], errors="coerce")
    bid = pd.to_numeric(df[COL_BID], errors="coerce")
    ask = pd.to_numeric(df[COL_ASK], errors="coerce")
    price = (bid + ask)/2.0
    is_call = df[COL_CP].astype(str).str.upper().str.strip().isin(["C","CALL","CALLS","1"]).to_numpy()

    # IV then Greeks
    iv = iv_newton(price.to_numpy(), S.to_numpy(), K.to_numpy(), T.to_numpy(), R, Q, is_call)
    Delta, Gamma, Theta, Vega = greeks(S.to_numpy(), K.to_numpy(), T.to_numpy(), R, Q, iv, is_call)

    out = df.copy()
    out["ImpliedVolatility"] = iv
    out["Delta"]  = Delta
    out["Gamma"]  = Gamma
    out["Theta"]  = Theta
    out["Theta_per_day"] = Theta / DAYS_PER_YEAR
    out["Vega"]   = Vega
    return out

# -------- run --------
files = [f for f in sorted(os.listdir(PICKLE_IN)) if f.lower().endswith(".pkl")]
for f in files:
    src = os.path.join(PICKLE_IN, f)
    try:
        out = process_file(src)
        dst = os.path.join(PICKLE_OUT, f)
        with open(dst, "wb") as fh: pickle.dump(out, fh, protocol=pickle.HIGHEST_PROTOCOL)
        nn = np.isfinite(out["ImpliedVolatility"]).sum()
        print(f"Saved {f}  rows={len(out)}  IV_nonnull={nn}")
    except Exception as e:
        print(f"[skip] {f}: {e}")


Saved SPX_OPTIONS_01011996.pkl  rows=5580  IV_nonnull=5580
Saved SPX_OPTIONS_01011997.pkl  rows=8378  IV_nonnull=8378
Saved SPX_OPTIONS_01011998.pkl  rows=8745  IV_nonnull=8745
Saved SPX_OPTIONS_01011999.pkl  rows=8654  IV_nonnull=8654
Saved SPX_OPTIONS_01012000.pkl  rows=10584  IV_nonnull=10584
Saved SPX_OPTIONS_01012001.pkl  rows=10863  IV_nonnull=10863
Saved SPX_OPTIONS_01012002.pkl  rows=9829  IV_nonnull=9829
Saved SPX_OPTIONS_01012003.pkl  rows=10702  IV_nonnull=10702
Saved SPX_OPTIONS_01012004.pkl  rows=10358  IV_nonnull=10358
Saved SPX_OPTIONS_01012005.pkl  rows=11424  IV_nonnull=11424
Saved SPX_OPTIONS_01012006.pkl  rows=12733  IV_nonnull=12733
Saved SPX_OPTIONS_01012007.pkl  rows=14718  IV_nonnull=14718
Saved SPX_OPTIONS_01012008.pkl  rows=22576  IV_nonnull=22576
Saved SPX_OPTIONS_01012009.pkl  rows=30300  IV_nonnull=30300
Saved SPX_OPTIONS_01012010.pkl  rows=35271  IV_nonnull=35271
Saved SPX_OPTIONS_01012011.pkl  rows=39620  IV_nonnull=39523
Saved SPX_OPTIONS_01012012.pkl  ro

In [38]:
#SANITY CHECK
import os, pickle, numpy as np, pandas as pd

OUT = "/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles/with_spx/with_iv_greeks"

def audit_one(path, nshow=5):
    with open(path, "rb") as f:
        df = pickle.load(f)

    required = ["SPX_AdjClose","ImpliedVolatility","Delta","Gamma","Theta","Vega","Strike","BestBid","BestOffer","Date","Expiration","CallPut"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        return {"file": os.path.basename(path), "rows": len(df), "status": f"Missing cols: {missing}"}

    # Basic counts
    iv = pd.to_numeric(df["ImpliedVolatility"], errors="coerce")
    delta = pd.to_numeric(df["Delta"], errors="coerce")
    gamma = pd.to_numeric(df["Gamma"], errors="coerce")
    theta = pd.to_numeric(df["Theta"], errors="coerce")
    vega  = pd.to_numeric(df["Vega"],  errors="coerce")

    # Time to expiry (years) to spot obvious parsing issues
    d = pd.to_datetime(pd.to_numeric(df["Date"], errors="coerce").round(0).astype("Int64").astype(str), format="%Y%m%d", errors="coerce")
    x = pd.to_datetime(pd.to_numeric(df["Expiration"], errors="coerce").round(0).astype("Int64").astype(str), format="%Y%m%d", errors="coerce")
    T = (x - d).dt.days.astype(float) / 365.0

    # Quick logical checks
    call_mask = df["CallPut"].astype(str).str.upper().isin(["C","CALL"])
    put_mask  = df["CallPut"].astype(str).str.upper().isin(["P","PUT"])
    price_mid = (pd.to_numeric(df["BestBid"], errors="coerce") + pd.to_numeric(df["BestOffer"], errors="coerce"))/2.0
    S = pd.to_numeric(df["SPX_AdjClose"], errors="coerce")
    K = pd.to_numeric(df["Strike"], errors="coerce")

    # Summaries
    out = {
        "file": os.path.basename(path),
        "rows": len(df),
        "IV non-null": int(iv.notna().sum()),
        "Δ in [-1,1] %": 100*float(((delta>=-1)&(delta<=1)).mean()) if len(df) else 0,
        "Γ >= 0 %": 100*float((gamma>=-1e-14).mean()) if len(df) else 0,
        "Vega >= 0 %": 100*float((vega>=-1e-14).mean()) if len(df) else 0,
        "T>0 %": 100*float((T>0).mean()) if len(df) else 0,
        "IV median": float(iv.median(skipna=True)) if iv.notna().any() else np.nan,
        "IV 1%..99%": (float(iv.quantile(0.01)), float(iv.quantile(0.99))) if iv.notna().sum()>5 else (np.nan,np.nan),
    }

    # Put-call sanity (only if you want a soft parity check)
    # Lower/upper no-arb bounds (rough)
    r, q = 0.045, 0.0
    disc_r = np.exp(-r*T)
    disc_q = np.exp(-q*T)
    call_lb = np.maximum(S*disc_q - K*disc_r, 0.0)
    put_lb  = np.maximum(K*disc_r - S*disc_q, 0.0)
    bad_calls = (call_mask) & (price_mid < call_lb - 1e-6)
    bad_puts  = (put_mask)  & (price_mid < put_lb  - 1e-6)
    out["Calls below LB %"] = 100*float(bad_calls.mean()) if bad_calls.size else 0
    out["Puts below LB %"]  = 100*float(bad_puts.mean())  if bad_puts.size else 0

    return out

def audit_folder(folder, limit=10):
    files = [os.path.join(folder,f) for f in sorted(os.listdir(folder)) if f.endswith(".pkl")]
    if not files:
        print("No files in", folder); return
    sample = files[:limit]
    rows = [audit_one(p) for p in sample]
    return pd.DataFrame(rows)

summary = audit_folder(OUT, limit=10)
print(summary)


                       file   rows  IV non-null  Δ in [-1,1] %  Γ >= 0 %  \
0  SPX_OPTIONS_01011996.pkl   5580         5580          100.0     100.0   
1  SPX_OPTIONS_01011997.pkl   8378         8378          100.0     100.0   
2  SPX_OPTIONS_01011998.pkl   8745         8745          100.0     100.0   
3  SPX_OPTIONS_01011999.pkl   8654         8654          100.0     100.0   
4  SPX_OPTIONS_01012000.pkl  10584        10584          100.0     100.0   
5  SPX_OPTIONS_01012001.pkl  10863        10863          100.0     100.0   
6  SPX_OPTIONS_01012002.pkl   9829         9829          100.0     100.0   
7  SPX_OPTIONS_01012003.pkl  10702        10702          100.0     100.0   
8  SPX_OPTIONS_01012004.pkl  10358        10358          100.0     100.0   
9  SPX_OPTIONS_01012005.pkl  11424        11424          100.0     100.0   

   Vega >= 0 %  T>0 %  IV median    IV 1%..99%  Calls below LB %  \
0        100.0  100.0   0.000001  (1e-06, 5.0)               0.0   
1        100.0  100.0   0.0

In [40]:
import numpy as np

def autoscale_strike(K: np.ndarray, S: np.ndarray, fname: str = "") -> np.ndarray:
    """
    If strikes are stored in 10x/100x/1000x units, scale them down.
    Strategy: bring median(K)/median(S) under ~10 by dividing by 10 repeatedly.
    Returns the scaled K. Prints the factor if any.
    """
    K = K.astype(float, copy=True)
    medK = np.nanmedian(K)
    medS = np.nanmedian(S)
    if not np.isfinite(medK) or not np.isfinite(medS) or medS <= 0:
        return K  # can't decide

    ratio = medK / medS
    factor = 1.0
    # If ratio is huge, scale down by 10s until it’s reasonable
    while ratio > 10 and factor < 1e9:
        factor *= 10.0
        ratio /= 10.0

    if factor > 1.0:
        print(f"[info] {fname}: scaling Strike by 1/{int(factor)} (medK/medS ratio fixed to ~{ratio:.2f})")
        K = K / factor
    return K

In [42]:
def process_file(path):
    with open(path,"rb") as f: df = pickle.load(f)
    if COL_S not in df.columns:
        raise KeyError(f"Missing {COL_S} — are you reading from with_spx/?")

    # Dates & time to expiry
    d  = to_dt_yyyymmdd(df[COL_D]); x = to_dt_yyyymmdd(df[COL_X])
    T  = (x - d).dt.days.astype(float) / DAYS_PER_YEAR

    # Inputs (as floats)
    S   = pd.to_numeric(df[COL_S],   errors="coerce").to_numpy()
    K   = pd.to_numeric(df[COL_K],   errors="coerce").to_numpy()
    bid = pd.to_numeric(df[COL_BID], errors="coerce").to_numpy()
    ask = pd.to_numeric(df[COL_ASK], errors="coerce").to_numpy()
    price  = (bid + ask) / 2.0
    is_call = df[COL_CP].astype(str).str.upper().str.strip().isin(["C","CALL","CALLS","1"]).to_numpy()

    # 🔧 NEW: auto-fix Strike scale if needed
    K = autoscale_strike(K, S, fname=os.path.basename(path))

    # IV then Greeks
    iv = iv_newton(price, S, K, T.to_numpy(), R, Q, is_call)
    Delta, Gamma, Theta, Vega = greeks(S, K, T.to_numpy(), R, Q, iv, is_call)

    out = df.copy()
    out["ImpliedVolatility"] = iv
    out["Delta"]  = Delta
    out["Gamma"]  = Gamma
    out["Theta"]  = Theta
    out["Theta_per_day"] = Theta / DAYS_PER_YEAR
    out["Vega"]   = Vega
    return out


In [43]:
with open("/Users/zakdjahed/Desktop/TAMID/Projects/2025-2026/OPPRCD_SPX/pickles/with_spx/with_iv_greeks/SPX_OPTIONS_01011996.pkl", "rb") as f:
    df = pickle.load(f)

df = df.copy()
df["Date"] = pd.to_datetime(df["Date"].astype(str), format="%Y%m%d", errors="coerce")
df["Expiration"] = pd.to_datetime(df["Expiration"].astype(str), format="%Y%m%d", errors="coerce")
df["T"] = (df["Expiration"] - df["Date"]).dt.days / 365.0

mid = (pd.to_numeric(df["BestBid"], errors="coerce") + pd.to_numeric(df["BestOffer"], errors="coerce"))/2
S   = pd.to_numeric(df["SPX_AdjClose"], errors="coerce")
K   = pd.to_numeric(df["Strike"], errors="coerce")
r, q = 0.045, 0.0
disc_r = np.exp(-r*df["T"])
disc_q = np.exp(-q*df["T"])

intrinsic_call = np.maximum(S*disc_q - K*disc_r, 0.0)
intrinsic_put  = np.maximum(K*disc_r - S*disc_q, 0.0)

print("Median mid price (calls):", float(mid[df["CallPut"].str.upper().eq("C")].median(skipna=True)))
print("Median intrinsic (calls):", float(intrinsic_call[df["CallPut"].str.upper().eq("C")].median(skipna=True)))
print("Median mid price (puts):", float(mid[df["CallPut"].str.upper().eq("P")].median(skipna=True)))
print("Median intrinsic (puts):", float(intrinsic_put[df["CallPut"].str.upper().eq("P")].median(skipna=True)))


Median mid price (calls): 37.3125
Median intrinsic (calls): 0.0
Median mid price (puts): 7.0
Median intrinsic (puts): 574160.8656797609
